# 01 — Baseline: does the evidence chain calibrate on source?

This notebook establishes the reference point before any stress test. We fit the strict
offline extractor and the survey calibration layer on the source training/dev calls, then
read out the sealed source holdout.

Two targets are reported side by side:

* **latent** — the generator's true satisfaction (available only because the data are synthetic);
* **survey** — the noisy simulated survey that the calibration layer was actually fitted to.

The gap between them is an audit finding in its own right: a pipeline can look calibrated
against surveys while drifting from the construct it claims to measure.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from callsat_audit.config import FIGURES_DIR, ROOT
from callsat_audit.extract import LexiconExtractor
from callsat_audit.generate import load_datasets
from callsat_audit.metrics import confusion, reliability_bins
from callsat_audit.pipeline import (
    build_frame,
    confusion_frame,
    extract_all,
    fit_calibration,
    probs_matrix,
    summarize_frame,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
datasets = load_datasets()
source = datasets["source"]

extractor = LexiconExtractor()
source_extractions = extract_all(source, extractor, condition="source")
bundle = fit_calibration(source, source_extractions)
print(
    f"calibration: temperature={bundle.temperature:.3f}, "
    f"conformal qhat={bundle.qhat:.3f} (alpha={bundle.alpha}), "
    f"source survey n={sum(c.survey_response is not None for c in source if c.split != 'holdout')}"
)

calibration: temperature=1.078, conformal qhat=0.893 (alpha=0.1), source survey n=227


In [2]:
frame = build_frame(source, source_extractions, bundle)
holdout = frame[frame["split"] == "holdout"].copy()

summary = pd.DataFrame(
    {
        "vs latent": summarize_frame(holdout, target="latent"),
        "vs survey": summarize_frame(holdout, target="survey"),
    }
)
summary_view = summary.loc[
    [
        "n",
        "qwk_raw",
        "qwk_ordinal",
        "qwk_isotonic",
        "mae_raw",
        "mae_ordinal",
        "mae_isotonic",
        "ece",
        "brier",
        "nll",
        "calibration_slope",
        "calibration_intercept",
        "mean_confidence",
        "large_error_rate",
    ]
].round(3)
summary_view.to_csv(ROOT / "results" / "runs" / "01_source_metrics.csv")
summary_view

,vs latent,vs survey
n,100.000,73.000
qwk_raw,0.718,0.502
qwk_ordinal,0.710,0.495
qwk_isotonic,0.717,0.549
mae_raw,0.625,0.835
mae_ordinal,0.631,0.814
mae_isotonic,0.624,0.796
ece,0.050,0.128
brier,0.625,0.737
nll,1.129,1.353


In [3]:
# Reliability diagram against the latent construct and against the survey.
latent = holdout.dropna(subset=["latent"])
survey = holdout.dropna(subset=["survey"])
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, subset, title in (
    (axes[0], latent, "vs latent satisfaction"),
    (axes[1], survey, "vs simulated survey"),
):
    xs, ys, ns = reliability_bins(subset["latent"] if title.startswith("vs latent") else subset["survey"], probs_matrix(subset))
    ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="perfect")
    ax.plot(xs, ys, marker="o", label="ordinal + temperature")
    ax.set_xlabel("mean confidence")
    ax.set_ylabel("empirical accuracy")
    ax.set_title(title)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc="lower right", fontsize=8)
fig.suptitle("Source holdout reliability")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_source_reliability.png", dpi=160)
plt.show()

/var/folders/g_/gj0vbxcd19b970fyngxp5k280000gn/T/ipykernel_27568/1645766263.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
cm = confusion_frame(holdout, target="latent")
fig, ax = plt.subplots(figsize=(4.6, 4.2))
image = ax.imshow(cm, cmap="Blues")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9)
ax.set_xticks(range(5), [1, 2, 3, 4, 5])
ax.set_yticks(range(5), [1, 2, 3, 4, 5])
ax.set_xlabel("predicted")
ax.set_ylabel("latent")
ax.set_title("Source holdout confusion")
fig.colorbar(image, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_source_confusion.png", dpi=160)
plt.show()

/var/folders/g_/gj0vbxcd19b970fyngxp5k280000gn/T/ipykernel_27568/394792162.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Evidence audit

`gold_turn_hit` is the fraction of facts whose cited turn overlaps a gold cue turn.
`quote_supported` checks that the extractor's quote actually appears in a transcript turn.
`missing_resolution_evidence` flags the call-eval failure mode where the scorer falls back
to a default because it could not find a resolution statement.

These are **oracle-assisted diagnostics** (they use the generator's gold turns). They tell us
where the chain breaks; they are not deployable runtime flags.

In [5]:
audit_table = pd.DataFrame(
    {
        "value": {
            "fact_accuracy_resolved": holdout["correct_resolved"].mean(),
            "fact_accuracy_effort": holdout["correct_effort"].mean(),
            "fact_accuracy_sentiment": holdout["correct_sentiment"].mean(),
            "fact_accuracy_empathy": holdout["correct_empathy"].mean(),
            "mean_fact_errors": holdout["fact_error_count"].mean(),
            "gold_turn_hit": holdout["gold_turn_hit"].mean(),
            "quote_supported": holdout["quote_supported"].mean(),
            "missing_resolution_evidence": holdout["missing_resolution_evidence"].mean(),
            "sentiment_resolution_conflict": holdout["sentiment_resolution_conflict"].mean(),
        }
    }
)
audit_table.round(3)

,value
fact_accuracy_resolved,0.790
fact_accuracy_effort,0.890
fact_accuracy_sentiment,0.940
fact_accuracy_empathy,0.870
mean_fact_errors,0.510
gold_turn_hit,0.848
quote_supported,1.000
missing_resolution_evidence,0.240
sentiment_resolution_conflict,0.060


In [6]:
# A single auditable trace: show the transcript, the extracted facts, and the cited spans.
candidate = next(
    (
        c
        for c in source
        if c.split == "holdout" and c.meta.get("implicit_resolution")
    ),
    next(c for c in source if c.split == "holdout"),
)
ex = source_extractions[candidate.call_id]
print(candidate.transcript())
print()
for name, fact in ex.facts.items():
    print(f"{name:10s} = {fact.value!r:12} conf={fact.confidence:.2f} turns={fact.evidence_turns} quote={fact.quotes}")
print("gold:", {k: candidate.gold_fact(k) for k in ("resolved", "effort", "sentiment", "empathy")})

[1] AGENT: Thanks for calling Meridian Support. This is Marcus. How can I help today?
[2] CUSTOMER: I'm still waiting on a refund I asked for two weeks ago. I've got about five minutes before my next meeting.
[3] AGENT: Could you verify the postcode on the account?
[4] CUSTOMER: It was $120.50, returned on the Wednesday.
[5] AGENT: Okay, I've found the record. Let me take a look.
[6] AGENT: I'll need to put you on a brief hold while I check that.
[7] AGENT: I'll have to transfer you to our billing team for this.
[8] AGENT: You'll need to call back tomorrow with the reference number.
[9] AGENT: Unfortunately, that's not something we can do.
[10] CUSTOMER: I'm not really happy with that.
[11] AGENT: Anything else I can do for you?
[12] CUSTOMER: That's everything. Thank you.

resolved   = 'partial'    conf=0.20 turns=() quote=()
effort     = 3            conf=0.85 turns=(6, 7, 8) quote=('brief hold', 'transfer you', 'call back tomorrow')
sentiment  = -1           conf=0.80 turns=(10,) qu

## Reading the baseline

The source holdout is the optimistic case: the extractor was written against these templates,
the calibration layer sees the same survey noise process, and no shift has occurred. Any
degradation we observe later is therefore attributable to the shift, not to an intrinsically
hard task.

If QWK against latent is meaningfully higher than QWK against survey, that is the **survey
ceiling** — a limit on what survey-anchored calibration can prove. If isotonic beats ordinal on
MAE but ordinal is needed for ECE and selective prediction, both layers should be kept for
different jobs.